# Pandera Exploration

In [ ]:
import pandas as pd
import pandera.pandas as pa             # när man jobbar med pandas DataFrames är det mycket rekommenderat att använda pandera.pandas istället för bara pandera

### DataFrameSchema

In [ ]:
# data att validera
df = pd.DataFrame({
    "station": ["ST1", "ST1", "ST3", "ST5"],
    "kunder": [1000, 400, 84, 2000],
    "avbrottstyp": ["planerat", "oplanerat", "oplanerat", "planerat"]
})

print(df)

In [ ]:
# sätt upp schema
schema = pa.DataFrameSchema({
    "station": pa.Column(
        str,
        pa.Check.isin(["ST1", "ST2", "ST3", "ST4", "ST5"])
    ),
    "kunder": pa.Column(int, pa.Check.ge(0)),
    "avbrottstyp": pa.Column(
        str,
        pa.Check.isin(["planerat", "oplanerat"])
    )
})

In [ ]:
validated_df = schema.validate(df)
print(validated_df)

Då datan stämde returneras df:en utan fel. 

In [ ]:
# data med fel 
df_fel = pd.DataFrame({
    "station": ["ST1", "ST1", "ST3", "ST6"],
    "kunder": [1000, 400, 84, 2000.4],
    "avbrottstyp": ["planerat", "oplanerat", "oplanerat", "planerat"]
})

print(df_fel)

In [ ]:
# validated_df_fel = schema.validate(df_fel)

Koden kraschar med ett SchemaError vid första felet (ST6 är inte en godkänd station)

### Dataframe Model

In [ ]:
# Definiera schema typ som en dataclass
class Schema(pa.DataFrameModel):
    station: str = pa.Field(isin=["ST1", "ST2", "ST3", "ST4", "ST5"])
    kunder: int = pa.Field(ge=0)
    avbrottstyp: str = pa.Field(isin=["planerat", "oplanerat"])

Schema.validate(df)

In [ ]:
# Test med felaktig data
# Schema.validate(df_fel)

Och det blir mycket riktigt ett SchemaError vid första felet.

### Informativa fel

In [ ]:
simple_schema = pa.DataFrameSchema({
    "voltage_level_kv": pa.Column(
        float,
        pa.Check(
            lambda x: 0.0 <= x <= 20.0,
            element_wise=True,
            error="range checker [0, 20]"
        )
    )
})

# datan bryter mot regeln
fail_check_df = pd.DataFrame({
    "voltage_level_kv": [-4.0, 0.4, 10.0, 20.0]
})

try:
    simple_schema(fail_check_df)
except pa.errors.SchemaError as exc:
    print(exc)

In [ ]:
# Med felaktigt kolumnnamn
wrong_column_df = pd.DataFrame({
    "duration_minutes": [5.6, 9.0]
})

try:
    simple_schema(wrong_column_df)
except pa.errors.SchemaError as exc:
    print(exc)